# `fred(series, start, end=None)` and `fred_series()`

FRED macro series from `macro.observations`, one value per date, oldest first. `fred_series()` is the map from each series id to its name, units, frequency and category. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

A series is named by its FRED id or by its name, in any case and spacing. Dates with no value (FRED's `.` gaps) are left out, never filled.

In [1]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.fred import FredSeries, FredValue
from arora_data.frames import to_df
from arora_data.sync import fred, fred_series


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The series
Every series the table holds, read from it on every call. The labels come from each series' latest row; `first_date` and `latest_date` are its first and latest dates with a value.

In [2]:
series = to_df(fred_series(), FredSeries)
print(f"{series.height} series")
series.group_by("category").agg(pl.len().alias("series"), pl.col("series_id").sort().str.join(" ").alias("ids")).sort("category")

115 series


category,series,ids
str,u32,str
"""Consumer""",11,"""DSPIC96 NONREVSL PCE PCEC96 PC…"
"""Credit""",10,"""ANFCI BAMLC0A0CM BAMLH0A0HYM2 …"
"""FX""",6,"""DEXCHUS DEXINUS DEXKOUS DEXMXU…"
"""Fiscal""",5,"""BOPGSTB FYOINT GFDEBTN MTSDS13…"
"""Growth""",12,"""AMTMNO BUSINV CP DGORDER GDPNO…"
…,…,…
"""Markets""",1,"""USEPUINDXD"""
"""Money""",13,"""BOGMBASE BUSLOANS M1SL M2SL MM…"
"""Rates""",14,"""AAA BAA DFEDTARL DFEDTARU DFF …"


## The map between names and ids
Two plain dicts from the same call. Every name is unique, so either one looks a series up.

In [3]:
by_name = {s.name: s.series_id for s in fred_series()}
by_id = {s.series_id: s.name for s in fred_series()}
assert len(by_name) == len(by_id)
print(by_name["10-Year TIPS Real Yield"], "|", by_id["THREEFYTP10"])
series.filter(pl.col("category") == "Rates").select("series_id", "name", "units", "frequency", "first_date", "latest_date")

DFII10 | Kim-Wright 10-Year Term Premium


series_id,name,units,frequency,first_date,latest_date
str,str,str,str,date,date
"""AAA""","""Moody's Aaa Corporate Yield""","""Percent""","""monthly""",1919-01-01,2026-10-01
"""BAA""","""Moody's Baa Corporate Yield""","""Percent""","""monthly""",1919-01-01,2026-10-01
"""DFEDTARL""","""Fed Funds Target Range, Lower""","""Percent""","""daily""",2008-12-16,2026-11-10
"""DFEDTARU""","""Fed Funds Target Range, Upper""","""Percent""","""daily""",2008-12-16,2026-11-10
"""DFF""","""Effective Fed Funds Rate""","""Percent""","""daily""",1954-07-01,2026-11-09
…,…,…,…,…,…
"""MORTGAGE15US""","""15-Year Fixed Mortgage Rate""","""Percent""","""weekly""",1991-08-30,2026-11-05
"""SOFR""","""Secured Overnight Financing Ra…","""Percent""","""daily""",2018-04-03,2026-11-09
"""T10YIE""","""10-Year Breakeven Inflation""","""Percent""","""daily""",2003-01-02,2026-11-10


## One series, by id or by name
The effective fed funds rate over the last two weeks. The id and the name, in any case and spacing, give the same rows.

In [4]:
two_weeks = today_et() - timedelta(days=14)
assert fred("DFF", two_weeks) == fred("dff", two_weeks) == fred("  effective FED funds rate ", two_weeks)
show(fred("DFF", two_weeks))

13 rows

series_id|name|date|value|units
---|---|---|---|---
DFF|Effective Fed Funds Rate|2026-10-28|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-10-29|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-10-30|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-10-31|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-11-01|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-11-02|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-11-03|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-11-04|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-11-05|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-11-06|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-11-07|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-11-08|3.880000|Percent
DFF|Effective Fed Funds Rate|2026-11-09|3.880000|Percent

## Gaps are left out
The 10-year TIPS yield in November 2025: 20 weekdays, but the bond market shut for Veterans Day (11-11) and Thanksgiving (11-27), so 18 values. And the October 2025 BLS report was never published (the shutdown), so monthly labor series skip 2025-10-01.

In [5]:
tips = fred("DFII10", date(2025, 11, 1), date(2025, 11, 30))
weekdays = [date(2025, 11, d) for d in range(1, 31) if date(2025, 11, d).weekday() < 5]
print(len(tips), "values; missing weekdays:", sorted(set(weekdays) - {v.date for v in tips}))
show(fred("U6RATE", date(2025, 8, 1), date(2025, 12, 31)))

18 values; missing weekdays: [datetime.date(2025, 11, 11), datetime.date(2025, 11, 27)]


4 rows

series_id|name|date|value|units
---|---|---|---|---
U6RATE|U-6 Underemployment Rate|2025-08-01|8.100000|Percent
U6RATE|U-6 Underemployment Rate|2025-09-01|8.100000|Percent
U6RATE|U-6 Underemployment Rate|2025-11-01|8.700000|Percent
U6RATE|U-6 Underemployment Rate|2025-12-01|8.400000|Percent

## How dates are set
Monthly and quarterly values are dated at the period's start. FYOINT, federal interest payments, is annual and dated at the fiscal year's end (09-30; 06-30 before 1976). IORB carries a value dated one day ahead, so an `end` past today is needed to see it.

In [6]:
print([v.date for v in fred("ECIALLCIV", date(2025, 1, 1))])
print([v.date for v in fred("FYOINT", date(1970, 1, 1), date(1980, 12, 31))])
today = today_et()
print("IORB to today:", fred("IORB", today - timedelta(days=3))[-1].date, "| to tomorrow:",
      fred("IORB", today - timedelta(days=3), today + timedelta(days=1))[-1].date)

[datetime.date(2025, 1, 1), datetime.date(2025, 4, 1), datetime.date(2025, 7, 1), datetime.date(2025, 10, 1), datetime.date(2026, 1, 1), datetime.date(2026, 4, 1), datetime.date(2026, 7, 1)]


[datetime.date(1970, 6, 30), datetime.date(1971, 6, 30), datetime.date(1972, 6, 30), datetime.date(1973, 6, 30), datetime.date(1974, 6, 30), datetime.date(1975, 6, 30), datetime.date(1976, 6, 30), datetime.date(1977, 9, 30), datetime.date(1978, 9, 30), datetime.date(1979, 9, 30), datetime.date(1980, 9, 30)]


IORB to today: 2026-11-11 | to tomorrow: 2026-11-12


## Full history in one call
From 1900 there is no row limit; a series starts at its own first value. The NBER recession indicator goes back the furthest.

In [7]:
history = to_df(fred("USREC", date(1900, 1, 1)), FredValue)
print(f"{history.height:,} months, {history['date'].min()} to {history['date'].max()}")
history.group_by(pl.col("date").dt.year() // 10 * 10).agg(pl.col("value").sum().alias("recession months")).sort("date").rename({"date": "decade"})

1,522 months, 1900-01-01 to 2026-10-01


decade,recession months
i32,"decimal[38,6]"
1900,48.000000
1910,54.000000
1920,49.000000
1930,52.000000
1940,19.000000
…,…
1980,22.000000
1990,8.000000
2000,26.000000


## Not here
Treasury yields are `treasury()` (FMP). GDP, CPI and unemployment are in `macro.economic_indicators` (FMP). Derived metrics such as T10Y2Y and T5YIFR are computed from stored series and never stored, so asking for them raises.

## Bad input
An unknown id or name, or a start after the end, raises a clear error before any values are read.

In [8]:
for args in [("DGS10", date(2026, 1, 1)), ("T5YIFR", date(2026, 1, 1)), ("fed funds", date(2026, 1, 1)), (None, date(2026, 1, 1)),
             ("DFF", date(2026, 1, 2), date(2026, 1, 1))]:
    try:
        fred(*args)
    except ValueError as e:
        print(e)

'DGS10' is not a stored FRED series: use an id or name from fred_series(), such as DFF or 'Effective Fed Funds Rate'


'T5YIFR' is not a stored FRED series: use an id or name from fred_series(), such as DFF or 'Effective Fed Funds Rate'


'fed funds' is not a stored FRED series: use an id or name from fred_series(), such as DFF or 'Effective Fed Funds Rate'


None is not a stored FRED series: use an id or name from fred_series(), such as DFF or 'Effective Fed Funds Rate'
start must be on or before end
